# Chapter 3: Inside the Transformer

> **Goal:** Understand the Transformer architecture — especially the Attention mechanism — without the math getting in the way.

---

## 🧠 Why Did Transformers Win?

Before Transformers (2017), we used **RNNs and LSTMs** to process sequences. They had two big problems:

1. **Sequential processing** — they read one word at a time, so training was slow
2. **Forgetting long-range dependencies** — after reading 100 words, they "forgot" word 1

Transformers solved both:
- Process all tokens **in parallel** (fast!)
- Every token can **directly attend** to every other token (no forgetting!)

---

## 🏗️ The Transformer Architecture

```
Input Tokens
    ↓
Token Embeddings + Positional Encoding
    ↓
┌─────────────────────────┐
│   Transformer Block ×N  │
│  ┌───────────────────┐  │
│  │  Multi-Head       │  │
│  │  Self-Attention   │  │
│  └───────────────────┘  │
│           +              │
│  Layer Normalization     │
│  ┌───────────────────┐  │
│  │  Feed-Forward     │  │
│  │  Network (MLP)    │  │
│  └───────────────────┘  │
│           +              │
│  Layer Normalization     │
└─────────────────────────┘
    ↓
Output (logits over vocabulary)
```

GPT models repeat this Transformer Block **12 to 96 times** — each repetition refines the representation.

---

## 🎯 The Attention Mechanism: The Core Idea

**Self-Attention** asks: *"For each word in this sentence, which OTHER words are most relevant to understanding it?"*

Example sentence: `"The animal didn't cross the street because it was too tired"`

When processing the word **"it"**, the model must figure out: does "it" refer to `animal` or `street`?

```
"it" ATTENDS TO:
  "animal"  → high attention (0.75)  ← correct answer
  "tired"   → medium attention (0.15)
  "street"  → low attention (0.05)
  other...  → low attention (0.05)
```

The model learned this from training — not from explicit rules!

---

## 🔑 Q, K, V: The Attention Recipe

Attention uses three learned projections for each token:

- **Q (Query):** "What am I looking for?" (the current token asking a question)
- **K (Key):** "What do I contain?" (all tokens advertising their content)
- **V (Value):** "What should I share?" (the actual information to aggregate)

The formula is:
```
Attention(Q, K, V) = softmax(Q × K^T / √d) × V

In plain English:
  1. Compute dot product of Q and all K vectors (similarity scores)
  2. Divide by √d to keep gradients stable
  3. Apply softmax to get a probability distribution
  4. Use those probabilities to take a weighted sum of V vectors
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def scaled_dot_product_attention(Q, K, V):
    """
    Pure NumPy implementation of Scaled Dot-Product Attention.
    
    Q: Query  (seq_len × d_k)
    K: Key    (seq_len × d_k)
    V: Value  (seq_len × d_v)
    """
    d_k = Q.shape[-1]  # Dimension of keys
    
    # Step 1: Compute similarity scores
    scores = Q @ K.T / np.sqrt(d_k)  # Shape: (seq_len × seq_len)
    
    # Step 2: Convert scores to probabilities (attention weights)
    # Softmax: e^x / sum(e^x)  — makes all weights sum to 1.0
    exp_scores = np.exp(scores - np.max(scores, axis=-1, keepdims=True))  # numerically stable
    attention_weights = exp_scores / exp_scores.sum(axis=-1, keepdims=True)
    
    # Step 3: Weighted sum of values
    output = attention_weights @ V
    
    return output, attention_weights


# --- Mini demo: 4 tokens, 8-dimensional embeddings ---
np.random.seed(42)

# Simulate a sentence: [The, cat, sat, mat]
tokens = ["The", "cat", "sat", "mat"]
seq_len = len(tokens)
d_model = 8   # embedding dimension (tiny for demo)
d_k = 4       # key/query dimension

# In a real Transformer, these weight matrices are LEARNED parameters
W_Q = np.random.randn(d_model, d_k)
W_K = np.random.randn(d_model, d_k)
W_V = np.random.randn(d_model, d_k)

# Simulated input embeddings (seq_len × d_model)
X = np.random.randn(seq_len, d_model)

# Project to Q, K, V
Q = X @ W_Q
K = X @ W_K
V = X @ W_V

# Run attention
output, attention_weights = scaled_dot_product_attention(Q, K, V)

# Visualize attention weights
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(attention_weights, cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Attention Weight')

ax.set_xticks(range(seq_len))
ax.set_yticks(range(seq_len))
ax.set_xticklabels(tokens)
ax.set_yticklabels(tokens)
ax.set_xlabel('Keys (what to attend TO)')
ax.set_ylabel('Queries (which token is attending)')
ax.set_title('Attention Weights Matrix\n(each row sums to 1.0)')

for i in range(seq_len):
    for j in range(seq_len):
        ax.text(j, i, f'{attention_weights[i, j]:.2f}',
                ha='center', va='center', fontsize=10,
                color='white' if attention_weights[i, j] > 0.6 else 'black')

plt.tight_layout()
plt.show()

## 🧩 Multi-Head Attention: Looking From Multiple Angles

A single attention head can only capture one type of relationship. **Multi-Head Attention** runs several attention operations in parallel, each looking for a different pattern:

```
Head 1: might track syntactic relationships (subject → verb)
Head 2: might track coreference ("it" → "cat")
Head 3: might track semantic similarity
Head 4: might track positional proximity
```

The outputs are concatenated and projected back:
```
MultiHead(Q,K,V) = Concat(head_1, ..., head_h) × W_O
```

GPT-2 (small) uses **12 attention heads**. GPT-3 uses **96 attention heads**.

---

## 📍 Positional Encoding: Giving Order to Tokens

Attention is **order-blind by default** — it treats all tokens the same regardless of position. We fix this by adding positional encoding to the embeddings.

Original Transformers used **sinusoidal encoding**: a unique pattern of sine/cosine waves for each position.

In [ ]:
def sinusoidal_positional_encoding(seq_len, d_model):
    """
    Create sinusoidal positional encoding matrix.
    Each row is the positional encoding for one position.
    """
    pos = np.arange(seq_len)[:, np.newaxis]       # (seq_len, 1)
    dim = np.arange(d_model)[np.newaxis, :]        # (1, d_model)
    
    # Even dimensions use sin, odd dimensions use cos
    angles = pos / np.power(10000, (2 * (dim // 2)) / d_model)
    encoding = np.where(dim % 2 == 0, np.sin(angles), np.cos(angles))
    
    return encoding

# Visualize positional encodings
seq_len = 50
d_model = 64
pe = sinusoidal_positional_encoding(seq_len, d_model)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Full heatmap
im = axes[0].imshow(pe, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding Dimension')
axes[0].set_ylabel('Position in Sequence')
axes[0].set_title('Positional Encodings (full matrix)')

# Show a few individual position vectors
for pos in [0, 5, 10, 20, 49]:
    axes[1].plot(pe[pos, :16], label=f'Position {pos}')
axes[1].set_xlabel('Embedding Dimension (first 16)')
axes[1].set_ylabel('Encoding Value')
axes[1].set_title('Positional Encoding Values for Different Positions')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Each position gets a unique 'fingerprint' vector.")
print("Adding this to the token embedding tells the model WHERE each token appears.")

## ✅ Chapter 3 Summary

| Concept | Key Point |
|---------|----------|
| Self-Attention | Every token can attend to every other token directly |
| Q, K, V | Query, Key, Value — the three projections that power attention |
| Scaled dot-product | Divide by √d to keep gradients stable during training |
| Multi-Head Attention | Run attention in parallel with different learned projections |
| Positional Encoding | Adds position information (since attention is order-blind) |
| Transformer Block | Multi-Head Attention + Feed-Forward + Layer Norm, repeated N times |

## 🔮 Up Next: Chapter 4 — Text Classification

We put these building blocks to work on the most common NLP task: **classifying text** (spam detection, sentiment analysis, topic labeling).